# Chunking

**Approach change:** Pure Python chunking - no Gemini API calls during indexing.

**Why:**
- Gemini for chunking = 83 large API calls = timeouts + slow + rate limits
- BGE-M3 embedding model is powerful enough to retrieve well from clean text chunks
- Gemini is used at QUERY time (answering), not at INDEX time (chunking)
- This is the standard production approach (LangChain, LlamaIndex all do it this way)

**Strategy:**
1. Recursive character splitter - respects paragraphs, sentences, words
2. Chunk size: 512 tokens (~2000 chars) with 20% overlap (~400 chars)
3. Each chunk gets: chunk_id, source, full_text, metadata
4. Metadata tagged automatically by keyword matching
5. Calculator tool descriptions added as special chunks

**Output:** `data/processed/chunks/all_chunks.json`

## 0. Setup

In [1]:
import json
from pathlib import Path
from datetime import datetime
import warnings; warnings.filterwarnings('ignore')

from tqdm import tqdm

NOTEBOOK_DIR = Path().resolve()
BACKEND_DIR = NOTEBOOK_DIR.parent
PROCESSED_DIR = BACKEND_DIR / 'data' / 'processed'
CHUNKS_DIR = PROCESSED_DIR / 'chunks'
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

PDF_DIR = PROCESSED_DIR / 'pdf_extracted'
EXCEL_DIR = PROCESSED_DIR / 'excel_extracted'
DOCX_DIR = PROCESSED_DIR / 'docx_extracted'

# Chunking config
CHUNK_SIZE = 2000   # characters (~500 tokens)
CHUNK_OVERLAP = 400    # characters (~100 tokens, 20% overlap)

# Files excluded from RAG chunking
VISUAL_REFERENCE_FILES = {'Body fat percentage visual reference guide PTC.pdf'}
CALCULATOR_EXCEL_FILES = {
    '1RM Calculator MennoHenselmans.com.xlsx',
    'Henselmans Energy intake calculator.xlsx',
    'Henselmans Energy Balance Calculator.xlsx',
    'Training volume calculator MennoHenselmans.com.xlsx',
}

print('Setup complete')
print(f'Chunk size: {CHUNK_SIZE} chars (~{CHUNK_SIZE//4} tokens)')
print(f'Chunk overlap: {CHUNK_OVERLAP} chars')

Setup complete
Chunk size: 2000 chars (~500 tokens)
Chunk overlap: 400 chars


## 1. Recursive Text Splitter

In [3]:
def split_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """
    Recursive character splitter.
    Tries to split on: paragraphs -> newlines -> sentences -> words -> characters.
    Guarantees chunks <= chunk_size with overlap between adjacent chunks.
    """
    separators = ['\n\n', '\n', '. ', '? ', '! ', ' ', '']

    def _split(text: str, seps: list[str]) -> list[str]:
        if not text.strip():
            return []
        if len(text) <= chunk_size:
            return [text.strip()]

        sep = seps[0]
        remaining_seps = seps[1:]

        if sep == '':
            # Hard split by character
            chunks = []
            for i in range(0, len(text), chunk_size - overlap):
                chunks.append(text[i:i + chunk_size])
            return chunks

        parts = text.split(sep)
        chunks, current = [], ''

        for part in parts:
            candidate = current + sep + part if current else part
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    # Recursively split current if still too large
                    if len(current) > chunk_size and remaining_seps:
                        chunks.extend(_split(current, remaining_seps))
                    else:
                        chunks.append(current.strip())
                current = part

        if current.strip():
            if len(current) > chunk_size and remaining_seps:
                chunks.extend(_split(current, remaining_seps))
            else:
                chunks.append(current.strip())

        return [c for c in chunks if c.strip()]

    raw_chunks = _split(text, separators)

    # Apply overlap: prepend end of previous chunk to each chunk
    if len(raw_chunks) <= 1 or overlap == 0:
        return raw_chunks

    overlapped = [raw_chunks[0]]
    for i in range(1, len(raw_chunks)):
        prev_tail = raw_chunks[i-1][-overlap:]
        overlapped.append(prev_tail + ' ' + raw_chunks[i])

    return overlapped


# Sanity check
_t = 'Hello world. ' * 200
_chunks = split_text(_t)
assert all(len(c) <= CHUNK_SIZE * 1.1 for c in _chunks), 'chunk too large'
print(f'Splitter OK - test: {len(_t)} chars -> {len(_chunks)} chunks')
print(f'Avg chunk size: {sum(len(c) for c in _chunks)//len(_chunks)} chars')

Splitter OK - test: 2600 chars -> 2 chunks
Avg chunk size: 1499 chars


## 2. Metadata Tagger

In [4]:
TOPIC_KEYWORDS = {
    'training': [
        'progressive overload','hypertrophy','strength','rep range','volume',
        'frequency','periodization','exercise selection','program','workout',
        'deload','intensity','RIR','RPE','failure','compound','isolation',
        'squat','deadlift','bench','press','sets','reps',
    ],
    'nutrition': [
        'protein','calorie','macro','carbohydrate','fat','diet','food',
        'meal','intake','deficit','surplus','TDEE','BMR','energy balance',
        'supplement','creatine','amino acid','nutrient','protein synthesis',
        'meal frequency','leucine',
    ],
    'recovery': [
        'sleep','recovery','deload','fatigue','stress','cortisol',
        'DOMS','soreness','rest','overtraining','overreaching','adaptation',
    ],
    'body_composition': [
        'body fat','lean mass','muscle mass','BF%','recomposition','recomp',
        'bulk','cut','bulking','cutting','fat loss','muscle gain',
        'body weight','weight loss','lean bulk',
    ],
    'program_design': [
        'program design','programming','split','PPL','upper lower','full body',
        'block','mesocycle','case study','client','intake form','assessment',
    ],
    'health': [
        'health','injury','pain','rehabilitation','lifestyle','adherence',
        'motivation','psychology','hormone','testosterone','insulin',
    ],
}

LEVEL_KEYWORDS = {
    'novice': ['novice','beginner','untrained','new to','starting'],
    'intermediate': ['intermediate','moderate experience'],
    'advanced': ['advanced','experienced','elite','competitive'],
}

GOAL_KEYWORDS = {
    'bulk': ['bulk','bulking','muscle gain','lean bulk','mass gain','surplus'],
    'cut': ['cut','cutting','fat loss','weight loss','deficit','lean out'],
    'maintain': ['maintain','maintenance','recomposition','recomp'],
}

CALCULATOR_KEYWORDS = {
    'energy_intake': ['TDEE','BMR','calorie','energy intake','caloric target'],
    'one_rep_max': ['1RM','one rep max','working weight'],
    'training_volume': ['sets per week','MEV','MAV','MRV','weekly sets'],
    'energy_balance': ['energy balance','body composition change'],
}

def tag_metadata(text: str, source: str, is_case_study: bool) -> dict:
    t = text.lower()

    scores = {cat: sum(1 for kw in kws if kw.lower() in t)
              for cat, kws in TOPIC_KEYWORDS.items()}
    topic = max(scores, key=scores.get)
    if scores[topic] == 0:
        topic = 'general'

    level = 'all'
    for lv, kws in LEVEL_KEYWORDS.items():
        if any(kw in t for kw in kws):
            level = lv
            break

    goals = [g for g, kws in GOAL_KEYWORDS.items() if any(kw in t for kw in kws)]
    goal = goals[0] if len(goals) == 1 else ('all' if not goals else 'multiple')

    calc_ctx = next(
        (c for c, kws in CALCULATOR_KEYWORDS.items() if any(kw.lower() in t for kw in kws)),
        None
    )

    return {
        'source': source,
        'topic_category': topic,
        'applies_to_level': level,
        'goal_relevance': goal,
        'is_case_study': is_case_study,
        'calculator_context': calc_ctx,
        'is_calculator_tool': False,
    }

print('Metadata tagger defined')

Metadata tagger defined


## 3. Load All Extracted Files

In [5]:
def load_all_docs() -> list[dict]:
    docs = []

    # PDFs
    for path in sorted(PDF_DIR.glob('*.json')):
        data  = json.loads(path.read_text(encoding='utf-8'))
        fname = data['filename']
        if fname in VISUAL_REFERENCE_FILES:
            continue
        is_cs = any(kw in fname.lower() for kw in ['case study','lifestyle case'])
        if data.get('full_text','').strip():
            docs.append({'filename': fname, 'source': data['source'],
                         'file_type': 'pdf', 'full_text': data['full_text'],
                         'is_case_study': is_cs})

    # Excel
    for path in sorted(EXCEL_DIR.glob('*.json')):
        data  = json.loads(path.read_text(encoding='utf-8'))
        fname = data['filename']
        if fname in CALCULATOR_EXCEL_FILES:
            continue
        is_cs = any(kw in fname.lower() for kw in ['case study','national','female powerlifter','ad libitum'])

        # Large Excel: sample first 150 rows per sheet
        text = data.get('full_text','')
        if not text.strip():
            continue
        if data.get('is_calculator'):
            continue
        docs.append({'filename': fname, 'source': data['source'],
                     'file_type': 'excel', 'full_text': text,
                     'is_case_study': is_cs})

    # Docx
    for path in sorted(DOCX_DIR.glob('*.json')):
        data = json.loads(path.read_text(encoding='utf-8'))
        fname = data['filename']
        if data.get('full_text','').strip():
            is_cs = 'case study' in fname.lower()
            docs.append({'filename': fname, 'source': data['source'],
                         'file_type': 'docx', 'full_text': data['full_text'],
                         'is_case_study': is_cs})

    return docs


docs = load_all_docs()
print(f'Documents loaded: {len(docs)}')
print(f'  Case studies: {sum(1 for d in docs if d["is_case_study"])}')
print(f'  Course material: {sum(1 for d in docs if not d["is_case_study"])}')
total_chars = sum(len(d["full_text"]) for d in docs)
print(f'  Total chars: {total_chars:,}')
print(f'  Est. chunks: ~{total_chars // (CHUNK_SIZE - CHUNK_OVERLAP):,}')

Documents loaded: 83
  Case studies: 30
  Course material: 53
  Total chars: 3,593,771
  Est. chunks: ~2,246


## 4. Chunk All Documents

In [6]:
all_chunks = []
chunk_stats = []

print(f'Chunking {len(docs)} documents...\n')

for doc in tqdm(docs, desc='Chunking'):
    fname = doc['filename']
    text = doc['full_text']
    source = doc['source']
    ftype = doc['file_type']
    is_cs = doc['is_case_study']
    stem = Path(source).stem

    raw_chunks = split_text(text)

    for j, chunk_text in enumerate(raw_chunks):
        if not chunk_text.strip():
            continue
        meta = tag_metadata(chunk_text, source, is_cs)
        chunk_id = f'{stem}__{j:04d}'
        all_chunks.append({
            'chunk_id': chunk_id,
            'source': source,
            'file_type': ftype,
            'text': chunk_text,
            'metadata': meta,
        })

    chunk_stats.append({
        'filename': fname,
        'file_type': ftype,
        'is_case_study': is_cs,
        'chunks': len(raw_chunks),
        'chars': len(text),
    })

print(f'\nChunking complete - {len(all_chunks):,} chunks from {len(docs)} documents')

Chunking 83 documents...



Chunking: 100%|██████████| 83/83 [00:00<00:00, 177.13it/s]


Chunking complete - 2,236 chunks from 83 documents


## 5. Add Calculator Tool Description Chunks

In [7]:
CALCULATOR_DESCRIPTIONS = [
    {
        'chunk_id': 'calculator__one_rep_max__description',
        'source': '1RM Calculator MennoHenselmans.com.xlsx',
        'file_type': 'excel',
        'text': (
            'One Rep Max (1RM) Calculator - Henselmans Method\n\n'
            'Estimates 1RM from a working set using the Epley formula: 1RM = weight x (1 + reps/30). '
            'Outputs appropriate training weight for any rep target (1-20 reps). '
            'Used to determine training intensity and prescribe loads across all rep ranges. '
            'Call calculate_1rm(weight, reps) from calculators.py during program generation. '
            'For bodyweight exercises (pull-ups, dips) use calculate_bodyweight_1rm().'
        ),
        'metadata': {'source': '1RM Calculator MennoHenselmans.com.xlsx',
                     'topic_category': 'training', 'applies_to_level': 'all',
                     'goal_relevance': 'all', 'is_case_study': False,
                     'calculator_context': 'one_rep_max', 'is_calculator_tool': True},
    },
    {
        'chunk_id': 'calculator__energy_intake__description',
        'source': 'Henselmans Energy intake calculator.xlsx',
        'file_type': 'excel',
        'text': (
            'Energy Intake Calculator - BMR, TDEE and Macros (Henselmans)\n\n'
            'Calculates BMR using Katch-McArdle formula (370 + 21.6 x LBM kg) - more accurate than Harris-Benedict. '
            'Multiplies by activity factor (sedentary 1.2 to very active 1.9) to get TDEE. '
            'Adjusts for goal: bulk +250 kcal, maintain 0, cut -500, aggressive cut -750. '
            'Protein: 2.0 g/kg during cut, 1.8 g/kg during bulk. Fat minimum 0.8 g/kg. Carbs fill remaining. '
            'Call calculate_energy_intake() from calculators.py. Always run validate_goal() first.'
        ),
        'metadata': {'source': 'Henselmans Energy intake calculator.xlsx',
                     'topic_category': 'nutrition', 'applies_to_level': 'all',
                     'goal_relevance': 'all', 'is_case_study': False,
                     'calculator_context': 'energy_intake', 'is_calculator_tool': True},
    },
    {
        'chunk_id': 'calculator__energy_balance__description',
        'source': 'Henselmans Energy Balance Calculator.xlsx',
        'file_type': 'excel',
        'text': (
            'Energy Balance Calculator - Expected Body Composition Change Rate (Henselmans)\n\n'
            'Calculates daily energy balance needed for target body composition change. '
            'Energy density constants: LBM = 1,817 kcal/kg, Fat mass = 9,081 kcal/kg. '
            'Weekly balance = (LBM change kg x 1817) + (FM change kg x 9081). Daily = weekly / 7. '
            'Use to cross-check that TDEE-based caloric target matches the intended rate of change. '
            'Call calculate_energy_balance(lbm_change_kg, fm_change_kg) from calculators.py.'
        ),
        'metadata': {'source': 'Henselmans Energy Balance Calculator.xlsx',
                     'topic_category': 'nutrition', 'applies_to_level': 'all',
                     'goal_relevance': 'all', 'is_case_study': False,
                     'calculator_context': 'energy_balance', 'is_calculator_tool': True},
    },
    {
        'chunk_id': 'calculator__training_volume__description',
        'source': 'Training volume calculator MennoHenselmans.com.xlsx',
        'file_type': 'excel',
        'text': (
            'Training Volume Calculator - Optimal Sets Per Week by Muscle Group (Henselmans)\n\n'
            'Provides optimal weekly set ranges per muscle group by training status. '
            'Novice/Intermediate/Advanced sets per week: '
            'Chest 10/12/16, Back 10/14/18, Shoulders 8/12/16, Biceps 6/10/14, Triceps 6/10/14, '
            'Quads 10/14/18, Hamstrings 8/10/14, Glutes 8/12/16, Calves 6/10/14, Abs 4/8/12. '
            'Female lower body +20%. Priority muscles +4 sets. Start at MEV, build to MAV. '
            'Call calculate_optimal_volume(training_status, is_female, priority_muscles) from calculators.py.'
        ),
        'metadata': {'source': 'Training volume calculator MennoHenselmans.com.xlsx',
                     'topic_category': 'training', 'applies_to_level': 'all',
                     'goal_relevance': 'all', 'is_case_study': False,
                     'calculator_context': 'training_volume', 'is_calculator_tool': True},
    },
]

all_chunks.extend(CALCULATOR_DESCRIPTIONS)
print(f'Calculator tool chunks added: {len(CALCULATOR_DESCRIPTIONS)}')
print(f'Total chunks: {len(all_chunks):,}')

Calculator tool chunks added: 4
Total chunks: 2,240


## 6. Handle Visual Reference PDF - Render as Images

In [8]:
import fitz

VISUAL_REF_DIR = PROCESSED_DIR / 'visual_reference'
VISUAL_REF_DIR.mkdir(exist_ok=True)

RAW_DIR = BACKEND_DIR / 'data' / 'raw'
pdf_name = 'Body fat percentage visual reference guide PTC.pdf'
pdf_path = next(RAW_DIR.rglob(pdf_name), None)

if pdf_path:
    doc = fitz.open(str(pdf_path))
    rendered = {}
    for page_num in range(doc.page_count):
        page = doc[page_num]
        pix = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
        out = VISUAL_REF_DIR / f'bf_visual_page_{page_num+1:02d}.png'
        pix.save(str(out))
        rendered[page_num + 1] = str(out)
    doc.close()

    manifest = {
        'source': pdf_name,
        'role': 'bf_visual_reference',
        'description': 'Passed to Gemini Vision alongside user photos for BF% assessment',
        'pages': rendered,
    }
    (VISUAL_REF_DIR / 'manifest.json').write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

    print(f'Visual reference: {len(rendered)} pages rendered as PNG')
    print(f'Manifest -> {VISUAL_REF_DIR / "manifest.json"}')
else:
    print(f'WARNING: {pdf_name} not found in {RAW_DIR}')
    rendered = {}

Visual reference: 67 pages rendered as PNG
Manifest -> D:\Cybernetic Gym Assistant\backend\data\processed\visual_reference\manifest.json


## 7. Statistics

In [9]:
from collections import Counter
import statistics

print(f'CHUNK STATISTICS')
print('=' * 55)
print(f'Total chunks: {len(all_chunks):,}')
print(f'  Document chunks: {len(all_chunks) - len(CALCULATOR_DESCRIPTIONS):,}')
print(f'  Calculator tools: {len(CALCULATOR_DESCRIPTIONS)}')

lengths = [len(c['text']) for c in all_chunks]
print(f'\nChunk length (chars):')
print(f'  Min: {min(lengths)}')
print(f'  Max: {max(lengths)}')
print(f'  Mean: {statistics.mean(lengths):.0f}')
print(f'  Median: {statistics.median(lengths):.0f}')

print(f'\nBy topic category:')
for t, n in Counter(c['metadata']['topic_category'] for c in all_chunks).most_common():
    print(f'  {t:<22}: {n:>5}')

print(f'\nBy file type:')
for t, n in Counter(c['file_type'] for c in all_chunks).most_common():
    print(f'  {t:<10}: {n:>5}')

print(f'\nCase study chunks: {sum(1 for c in all_chunks if c["metadata"]["is_case_study"])}')
print(f'Calculator context: {sum(1 for c in all_chunks if c["metadata"]["calculator_context"])}')

print(f'\nTop files by chunk count:')
file_counts = Counter(c['source'] for c in all_chunks if not c['metadata'].get('is_calculator_tool'))
for src, n in file_counts.most_common(10):
    print(f'  {Path(src).name[:52]:<52}: {n:>4} chunks')

CHUNK STATISTICS
Total chunks: 2,240
  Document chunks: 2,236
  Calculator tools: 4

Chunk length (chars):
  Min: 21
  Max: 2401
  Mean: 1977
  Median: 2126

By topic category:
  training              :   996
  nutrition             :   944
  recovery              :    84
  general               :    83
  health                :    66
  body_composition      :    37
  program_design        :    30

By file type:
  pdf       :  2091
  excel     :   111
  docx      :    38

Case study chunks: 220
Calculator context: 714

Top files by chunk count:
  Protein PTC 2022.pdf                                :  132 chunks
  Adherence PTC 2022.pdf                              :  120 chunks
  Exercise Selection PTC 2022 (1).pdf                 :  109 chunks
  Health science and food choices PTC 2022.pdf        :  107 chunks
  Energy PTC 2022.pdf                                 :  106 chunks
  Supplements PTC 2022.pdf                            :  103 chunks
  Dietary fat PTC 2022.pdf               

## 8. Save

In [11]:
output = {
    'created_at': datetime.now().isoformat(),
    'total_chunks': len(all_chunks),
    'document_chunks': len(all_chunks) - len(CALCULATOR_DESCRIPTIONS),
    'calculator_tool_chunks': len(CALCULATOR_DESCRIPTIONS),
    'chunk_size_chars': CHUNK_SIZE,
    'chunk_overlap_chars': CHUNK_OVERLAP,
    'chunks': all_chunks,
}

out_path = CHUNKS_DIR / 'all_chunks.json'
out_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding='utf-8')
size_mb = out_path.stat().st_size / (1024 * 1024)

print('=' * 55)
print('  NOTEBOOK 04 — COMPLETE')
print('=' * 55)
print(f'  Total chunks: {len(all_chunks):,}')
print(f'  File size: {size_mb:.1f} MB')
print(f'  Output: {out_path}')

  NOTEBOOK 04 — COMPLETE
  Total chunks: 2,240
  File size: 5.3 MB
  Output: D:\Cybernetic Gym Assistant\backend\data\processed\chunks\all_chunks.json
